In [1]:
import argparse
from itertools import product

import numpy as np

from cc2cc import add_args, cc, ucc
from cc2cc.utils import Grid, gen_mole, print_computer_info
from cc2cc.utils.env_var import DATA_PATH
from cc2cc.utils.parser import gen_name_args

origin_mol_str_list = [
    # "molecule1-W4_11",
    # "molecule2-W4_11",
    "molecule3-W4_11",
    # "molecule4-W4_11",
    # "molecule5-W4_11",
]
name_mol_str_list = [
    # "molecule1",
    # "molecule2",
    "molecule3",
    # "molecule4",
]
name_mol_str_exclude_list = [
    "W4_11-propane",  # 3
    "molecule1-W4_11",
    "molecule2-W4_11",
    "molecule3-W4_11",
    "molecule4-W4_11",
    "molecule5-W4_11",
    "molecule1-ACC24",
    "molecule1-GAPS",
    "molecule1-GW100",
    "molecule1-MRADC",
    "molecule1-S30L",
    "molecule2-ACC24",
    "molecule2-GAPS",
    "molecule2-GW100",
    "molecule2-MRADC",
    "molecule2-S30L",
    "molecule3-ACC24",
    "molecule3-GAPS",
    "molecule3-GW100",
    "molecule3-MRADC",
    "molecule3-S30L",
    "molecule4-ACC24",
    "molecule4-GAPS",
    "molecule4-GW100",
    "molecule4-MRADC",
    "molecule4-S30L",
]

mol_elements_dict = {}
len_elements_dict = {}

if __name__ == "__main__":
    error_molecule = []

    name_mol_list = gen_name_args(name_mol_str_list, "filtered_gmtkn-cc-pVDZ")
    name_mol_exclude_list = gen_name_args(
        name_mol_str_exclude_list, "filtered_gmtkn-cc-pVDZ", if_exclude=True
    )
    name_mol_list = [mol for mol in name_mol_list if mol not in name_mol_exclude_list]

    origin_mol_list = gen_name_args(origin_mol_str_list, "filtered_gmtkn-cc-pVDZ")
    name_mol_list = origin_mol_list + name_mol_list

    error_molecule = []
    print(f"Name Molecule List: {name_mol_list}")

    for name_mol in name_mol_list:
        try:
            mol = gen_mole(
                name_mol,
                0,
                1,
                0,
                "cc-pVDZ",
                True,
                "filtered_gmtkn-cc-pVDZ",
                if_rotate=True,
                if_rotate_random=False,
                solve_symmetry=True,
                verbose=1,
            )

            mol_elements = list(np.array(mol.elements))
            mol_atom_coords = list(mol.atom_coords())
            mol_atom_coords.append(name_mol)
            # print(mol_atom_coords)
            mol_elements.extend([mol.charge, mol.spin])
            mol_elements_str = "-".join(map(str, mol_elements))
            if mol_elements_str not in mol_elements_dict:
                mol_elements_dict[mol_elements_str] = [mol_atom_coords]
            else:
                mol_elements_dict[mol_elements_str].append(mol_atom_coords)
            len_elements_dict[mol_elements_str] = len(list(np.array(mol.elements)))

        except (ValueError, RuntimeError) as e:
            print(f"ERROR: {name_mol}")
            print(e)
            error_molecule.append(name_mol)
            print(f"Error molecule: {error_molecule}")
        finally:
            print(f"Processed: {name_mol}")
        print()

    print(f"Error molecule: {error_molecule}")

Name Molecule List: ['W4_11-becl2', 'W4_11-bef2', 'W4_11-ccl2', 'W4_11-cf2', 'W4_11-cl2o', 'W4_11-clcn', 'W4_11-cloo', 'W4_11-co2', 'W4_11-cs2', 'W4_11-f2o', 'W4_11-fo2', 'W4_11-n2o', 'W4_11-no2', 'W4_11-o3', 'W4_11-oclo', 'W4_11-ocs', 'W4_11-s2o', 'W4_11-s3', 'W4_11-so2', 'W4_11-bhf2', 'W4_11-c-hono', 'W4_11-c-hooo', 'W4_11-hccf', 'W4_11-hcno', 'W4_11-hcof', 'W4_11-hnco', 'W4_11-hnnn', 'W4_11-hocn', 'W4_11-honc', 'W4_11-t-hono', 'W4_11-t-hooo', 'W4_11-ch2f2', 'W4_11-dioxirane', 'W4_11-formic', 'W4_11-ketene', 'W4_11-oxirene', 'W4_11-c2h3f', 'W4_11-acetaldehyde', 'W4_11-allene', 'W4_11-oxirane', 'W4_11-propyne', 'W4_11-c2h5f', 'W4_11-ethanol', 'W4_11-propene', 'W4_11-propane', 'AHB21-11A', 'AHB21-12A', 'AHB21-13A', 'AHB21-14A', 'BH76-n2o', 'BH9-04_10R2', 'BH9-04_11R2', 'BH9-04_12R2', 'BH9-04_13R2', 'BH9-04_15R2', 'BH9-04_9R2', 'BH9-09_13R1', 'BHPERI-13r_1', 'DC13-o3', 'FH51-CO2', 'G2RC-40', 'G2RC-56', 'G2RC-68', 'G2RC-73', 'INV24-SO2', 'INV24-SO2_TS', 'RG18-ar3', 'RG18-kr3', 'RG18-ne3'

In [2]:
for mol_elements_name, mol_elements in mol_elements_dict.items():
    # print(f"Processing {mol_elements_name}")
    # print(f"{mol_elements}")

    if len_elements_dict[mol_elements_name] > 12:
        print("Skip large molecules (>12 atoms)\n")
        continue

    identifiables = [0]
    for i_elements in range(1, len(mol_elements)):
        distance_list = np.zeros(len(identifiables))
        for iter, identifiable in enumerate(identifiables):
            for i_element in range(len(mol_elements[i_elements]) - 1):
                distance_list[iter] = max(
                    np.linalg.norm(
                        mol_elements[i_elements][i_element]
                        - mol_elements[identifiable][i_element]
                    ),
                    distance_list[iter],
                )
        if np.all(distance_list > 0.1):
            identifiables.append(i_elements)

    # print(f"==={distance_list}===")
    for identifiable in identifiables:
        print(f'"{mol_elements[identifiable][-1]}",')
        # print(
        #     f"{np.array2string(np.array(mol_elements[identifiable][:-1]), formatter={'float': '{: .2f}'.format})}"
        # )
    print("")

"W4_11-becl2",

"W4_11-bef2",

"W4_11-ccl2",

"W4_11-cf2",

"W4_11-cl2o",

"W4_11-clcn",

"W4_11-cloo",
"W4_11-oclo",

"W4_11-co2",

"W4_11-cs2",

"W4_11-f2o",

"W4_11-fo2",

"W4_11-n2o",
"G2RC-68",

"W4_11-no2",

"W4_11-o3",

"W4_11-ocs",

"W4_11-s2o",

"W4_11-s3",

"W4_11-so2",
"INV24-SO2_TS",

"W4_11-bhf2",

"W4_11-c-hono",
"W4_11-t-hono",

"W4_11-c-hooo",
"W4_11-t-hooo",

"W4_11-hccf",
"HAL59-FCCH",

"W4_11-hcno",
"W4_11-hnco",
"W4_11-hocn",
"W4_11-honc",

"W4_11-hcof",

"W4_11-hnnn",
"BH9-04_10P2",

"W4_11-ch2f2",

"W4_11-dioxirane",
"W4_11-formic",
"G2RC-104",
"WCPT18-ts1",

"W4_11-ketene",
"W4_11-oxirene",

"W4_11-c2h3f",
"G2RC-121",

"W4_11-acetaldehyde",
"W4_11-oxirane",
"G2RC-113",
"S66-59",
"WCPT18-reac3",
"WCPT18-ts3",

"W4_11-allene",
"W4_11-propyne",
"BH9-09_4R1",
"ISO34-P2",

"W4_11-c2h5f",
"RSE43-E9",

"W4_11-ethanol",
"BH9-04_38P2",
"INV24-Ether",
"INV24-Ether_TS",
"ISO34-E24",
"PA26-ethanol",
"RSE43-E10",

"W4_11-propene",
"BH9-02_62R2",
"BH9-06_19R1",
"G2RC-82",
"ISO

In [4]:
len(origin_mol_list)

22